# Name That LLM - Improved Baseline (Ensemble Embeddings)

**Approach:**
1. Ensemble multiple sentence transformer models for richer embeddings
2. Standardize concatenated embeddings
3. Logistic Regression with balanced class weights
4. Stratified 5-fold CV evaluation

**Models used:**
- `all-mpnet-base-v2` (768 dim) - strong general purpose
- `all-MiniLM-L6-v2` (384 dim) - fast, complementary features
- Total: 1152 dimensions

**Kaggle Settings:**
- Accelerator: GPU T4 x2
- Internet: Off (models cached after first run)

In [ ]:
# Install dependencies
%pip install -q sentence-transformers scikit-learn pandas numpy joblib torch

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
import torch
import joblib
import warnings
warnings.filterwarnings('ignore')

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Load data
train_df = pd.read_csv('/kaggle/input/name-that-llm/train.csv')
test_df = pd.read_csv('/kaggle/input/name-that-llm/test.csv')
sample_sub = pd.read_parquet('/kaggle/input/name-that-llm/sample_submission.parquet')

print(f"Train: {train_df.shape}, Test: {test_df.shape}")
print(f"Models: {train_df['model'].nunique()}")
print(train_df['model'].value_counts().sort_index())

In [ ]:
# Ensemble of embedding models
model_names = [
    "sentence-transformers/all-mpnet-base-v2",
    "sentence-transformers/all-MiniLM-L6-v2",
]

all_train_emb = []
all_test_emb = []

for name in model_names:
    print(f"Loading {name}...")
    model = SentenceTransformer(name, device=device)
    
    train_emb = model.encode(
        train_df['text'].tolist(),
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    test_emb = model.encode(
        test_df['text'].tolist(),
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    all_train_emb.append(train_emb)
    all_test_emb.append(test_emb)
    print(f"  Shapes: train={train_emb.shape}, test={test_emb.shape}")

In [ ]:
# Concatenate embeddings
X_train = np.hstack(all_train_emb)
X_test = np.hstack(all_test_emb)
print(f"Combined: train={X_train.shape}, test={X_test.shape}")

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Labels
le = LabelEncoder()
y_train = le.fit_transform(train_df['model'])

In [ ]:
# Cross-validation
clf = LogisticRegression(
    max_iter=3000,
    C=1.0,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    solver='lbfgs'
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_scaled, y_train)):
    clf.fit(X_train_scaled[tr_idx], y_train[tr_idx])
    pred = clf.predict(X_train_scaled[val_idx])
    score = balanced_accuracy_score(y_train[val_idx], pred)
    cv_scores.append(score)
    print(f"Fold {fold+1}: {score:.4f}")

print(f"\nCV Balanced Acc: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

In [ ]:
# Full training
clf.fit(X_train_scaled, y_train)

# Test predictions
test_pred = clf.predict(X_test_scaled)
test_labels = le.inverse_transform(test_pred)

submission = pd.DataFrame({'id': test_df['id'], 'model': test_labels})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(submission)

In [ ]:
# Save artifacts
joblib.dump(clf, '/kaggle/working/classifier.pkl')
joblib.dump(le, '/kaggle/working/label_encoder.pkl')
joblib.dump(scaler, '/kaggle/working/scaler.pkl')
print("Artifacts saved.")